# Applied Deep Learning — ASL Image Classification

In this lab, you will build and train a PyTorch neural network to classify static hand-sign images representing letters from the **American Sign Language (ASL) alphabet**.


### Dataset
This practical uses the **Sign Language MNIST** dataset originally published on Kaggle by *datamunge*. The dataset is distributed under **CC0: Public Domain**. It contains 28×28 grayscale images of hand signs. The static-image dataset excludes **J** and **Z**, because those signs involve motion.

### Learning outcomes
By the end of this lab, you should be able to:

- load a custom image dataset stored in CSV format;
- inspect and visualise image data and class labels;
- normalise pixel values;
- create a custom PyTorch `Dataset` and `DataLoader`;
- build a fully connected neural network for multi-class classification;
- train and validate a PyTorch model using GPU acceleration when available;
- calculate loss and classification accuracy;
- identify the difference between training and validation performance;
- recognise the signs of **overfitting**.


## 1. Import libraries and select the compute device

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Selected device:', device)

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 2. Obtain the Sign Language MNIST dataset

The two files required are:

```text
sign_mnist_train.csv
sign_mnist_test.csv
```


## 3. Load the CSV data

Each row represents one 28×28 grayscale image:

- one `label` column;
- 784 pixel-value columns (`28 × 28 = 784`).

Pixel values initially range from `0` to `255`.


In [ ]:
train_df = pd.read_csv("sign_mnist_train.csv")
valid_df = pd.read_csv("sign_mnist_valid.csv")

print('Training shape  :', train_df.shape)
print('Validation shape:', valid_df.shape)
train_df.head()


### 3.1 Separate labels and image pixels

In [ ]:
y_train_raw = train_df['label'].to_numpy()
y_valid_raw = valid_df['label'].to_numpy()

x_train = train_df.drop(columns='label').to_numpy()
x_valid = valid_df.drop(columns='label').to_numpy()

print('x_train:', x_train.shape)
print('y_train:', y_train_raw.shape)
print('x_valid:', x_valid.shape)
print('y_valid:', y_valid_raw.shape)
print('Pixel range:', x_train.min(), 'to', x_train.max())


### 3.2 Remap labels to contiguous class indices

The original Sign Language MNIST labels correspond to alphabet positions, while the moving letters **J** and **Z** are absent. For PyTorch `CrossEntropyLoss`, it is convenient to map the remaining labels to contiguous class indices `0 ... 23`.

We keep a reverse mapping so that class indices can still be displayed as letters.


In [ ]:
unique_labels = sorted(np.unique(y_train_raw))
label_to_index = {label: idx for idx, label in enumerate(unique_labels)}
index_to_label = {idx: label for label, idx in label_to_index.items()}

y_train = np.array([label_to_index[v] for v in y_train_raw], dtype=np.int64)
y_valid = np.array([label_to_index[v] for v in y_valid_raw], dtype=np.int64)

n_classes = len(unique_labels)

print('Number of classes:', n_classes)
print('Mapped label range:', y_train.min(), 'to', y_train.max())


## 4. Visualise sample images

In [ ]:
plt.figure(figsize=(16, 6))

num_images = 12
for i in range(num_images):
    image = x_train[i].reshape(28, 28)
    letter = index_to_label[y_train[i]]

    plt.subplot(3, 4, i + 1)
    plt.imshow(image, cmap='gray')
    plt.title(f'Label: {letter}')
    plt.axis('off')

plt.tight_layout()
plt.show()


## 5. Normalise the image data

Neural networks generally train more effectively when input values are kept within a smaller numerical range. We scale the pixel values from `0–255` to `0–1`.


In [ ]:
x_train = x_train.astype('float32') / 255.0
x_valid = x_valid.astype('float32') / 255.0

print('Training pixel range:', x_train.min(), 'to', x_train.max())


## 6. Create a custom PyTorch Dataset

A PyTorch `Dataset` defines how individual samples are retrieved. The `DataLoader` then groups samples into mini-batches.

For portability and good memory practice, this lab keeps the full dataset in CPU memory and moves **each mini-batch** to the GPU during training. This pattern scales better to larger datasets than storing the entire dataset on GPU memory.


In [ ]:
class ASLDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


BATCH_SIZE = 32

train_data = ASLDataset(x_train, y_train)
valid_data = ASLDataset(x_valid, y_valid)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False)

print('Training samples  :', len(train_data))
print('Validation samples:', len(valid_data))
print('Training batches  :', len(train_loader))
print('Validation batches:', len(valid_loader))


### 6.1 Inspect one mini-batch

In [ ]:
batch_x, batch_y = next(iter(train_loader))

print('Image batch shape:', batch_x.shape)
print('Label batch shape:', batch_y.shape)
print('First few labels :', batch_y[:10].tolist())


## 7. Build a fully connected classifier

### Exercise

Create a network with:

- `Flatten()`;
- a fully connected layer with **512 neurons** and ReLU;
- a second fully connected layer with **512 neurons** and ReLU;
- an output layer with one neuron per ASL class.

Recall that the images contain `28 × 28 = 784` input values.


In [ ]:
input_size = 28 * 28

# TODO: Complete the model.
model = nn.Sequential(
    # nn.Flatten(),
    # ...
)


### Solution

Run this cell after attempting the exercise.


In [ ]:
input_size=28*28
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 512),
    nn.ReLU(),
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Linear(512, n_classes)
).to(device)

print(model)
print('Model device:', next(model.parameters()).device)


## 8. Loss function and optimizer

For multi-class classification with integer class labels, use:

```python
nn.CrossEntropyLoss()
```

This loss expects **raw logits** from the final layer, so do **not** add a `Softmax` layer to the model.


In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-3)


## 9. Training and validation functions

The training loop performs:

1. forward pass;
2. loss calculation;
3. gradient reset;
4. backpropagation;
5. optimizer update;
6. metric accumulation.

Validation uses `model.eval()` and `torch.no_grad()` because model parameters should not be updated.


In [ ]:
def train(model=model):
    loss = 0
    total_correct=0
    total_samples=0
    accuracy = 0
    avg_loss = 0

    model.train()
    for x, y in train_loader:
        output = model(x)
        optimizer.zero_grad()
        batch_loss = loss_function(output, y)
        batch_loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        loss += batch_loss.item() * batch_size
        total_correct += (output.argmax(dim=1) == y).sum().item()
        total_samples += batch_size
        avg_loss = loss / total_samples
        accuracy = total_correct / total_samples

    print('Train - Loss: {:.4f} Accuracy: {:.4f}'.format(avg_loss, accuracy))
    return avg_loss, accuracy

In [ ]:
def validate(model=model):
    loss = 0
    total_correct=0
    total_samples=0
    accuracy = 0
    avg_loss = 0

    model.eval()
    with torch.no_grad():
        for x, y in valid_loader:
            output = model(x)

            batch_size = y.size(0)
            loss += loss_function(output, y).item()*batch_size
            total_correct += (output.argmax(dim=1) == y).sum().item()
            total_samples += batch_size
            avg_loss = loss / total_samples
            accuracy = total_correct / total_samples

    print('Valid - Loss: {:.4f} Accuracy: {:.4f}'.format(avg_loss, accuracy))
    return avg_loss, accuracy

## 10. Train the model

Start with a modest number of epochs. You can increase this later when exploring overfitting.


In [ ]:
EPOCHS = 10
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train(model)
    valid_loss, valid_acc = validate(model)

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'valid_loss': valid_loss,
        'valid_acc': valid_acc,
    })

    print(
        f'Epoch {epoch:02d}/{EPOCHS} | '
        f'Train loss: {train_loss:.4f} | Train acc: {train_acc*100:.2f}% | '
        f'Valid loss: {valid_loss:.4f} | Valid acc: {valid_acc*100:.2f}%'
    )


## 11. Plot the learning curves

Learning curves make it easier to compare training and validation behaviour over time.


In [ ]:
history_df = pd.DataFrame(history)

plt.figure(figsize=(7, 4))
plt.plot(history_df['epoch'], history_df['train_loss'], marker='o', label='Training loss')
plt.plot(history_df['epoch'], history_df['valid_loss'], marker='o', label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss by Epoch')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history_df['epoch'], history_df['train_acc'] * 100, marker='o', label='Training accuracy')
plt.plot(history_df['epoch'], history_df['valid_acc'] * 100, marker='o', label='Validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy by Epoch')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 12. Discussion — is the model overfitting?

Compare the training and validation curves.

Consider:

1. Does training accuracy continue to increase?
2. Does validation accuracy improve at the same rate?
3. Does validation loss begin to rise while training loss continues to fall?
4. What does this tell you about the model's ability to generalise to unseen images?

A widening gap between training and validation performance is a common sign of **overfitting**.

In later labs, you will address this using techniques such as:

- convolutional neural networks (CNNs);
- data augmentation;
- regularisation;
- transfer learning.


## 13. Optional — inspect some predictions

In [ ]:
model.eval()

x_batch, y_batch = next(iter(valid_loader))
x_gpu = x_batch.to(device)

with torch.no_grad():
    output = model(x_gpu)
    pred = output.argmax(dim=1).cpu()

plt.figure(figsize=(14, 6))

for i in range(min(12, len(x_batch))):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_batch[i].reshape(28, 28), cmap='gray')

    true_letter = index_to_label[int(y_batch[i])]
    pred_letter = index_to_label[int(pred[i])]

    plt.title(f'True: {true_letter} | Pred: {pred_letter}')
    plt.axis('off')

plt.tight_layout()
plt.show()


## 14. Summary

In this lab you:

- loaded a custom image dataset from CSV files;
- converted raw labels into contiguous PyTorch class indices;
- normalised image data;
- created a custom `Dataset` and `DataLoader`;
- built a fully connected neural network;
- trained and validated the model on CPU or GPU;
- plotted learning curves;
- examined the gap between training and validation performance.

### Reflection

Why might a fully connected network struggle to generalise well on image data even when its training accuracy becomes very high?

This question motivates the next major topic: **Convolutional Neural Networks (CNNs)**.


## Dataset attribution

**Sign Language MNIST** — Kaggle dataset by *datamunge*  
License: **CC0: Public Domain**  
Dataset page: `https://www.kaggle.com/datasets/datamunge/sign-language-mnist`

The notebook code and instructional material in this practical are independent course materials for the Applied Deep Learning module.
